# Inference Engineering Assignment

I am still learning AI/ML, so I kept the code simple and wrote the notes in my own words.

## Task 1: Quantization (float32 to int8)

int8 can only hold whole numbers from -128 to 127. So I squeeze the float range [min, max] into that range.
- scale = (max - min) / 255
- zero point = the int8 number that stands for 0.0
- quantize: round(x / scale) + zero point
- dequantize: (q - zero point) * scale

In [1]:
def quantize(values):
    low = min(values)
    high = max(values)
    scale = (high - low) / 255
    zero_point = round(-128 - low / scale)
    result = []
    for x in values:
        q = round(x / scale) + zero_point
        q = max(-128, min(127, q))
        result.append(q)
    return result, scale, zero_point

def dequantize(q_values, scale, zero_point):
    return [(q - zero_point) * scale for q in q_values]

In [2]:
data = [-1.5, -0.3, 0.0, 0.7, 2.1, 3.4]

q_values, scale, zero_point = quantize(data)
back = dequantize(q_values, scale, zero_point)

print("original :", data)
print("int8     :", q_values)
print("back     :", [round(b, 3) for b in back])
print("scale    :", round(scale, 4), " zero point:", zero_point)

original : [-1.5, -0.3, 0.0, 0.7, 2.1, 3.4]
int8     : [-128, -66, -50, -14, 59, 127]
back     : [-1.499, -0.307, 0.0, 0.692, 2.095, 3.401]
scale    : 0.0192  zero point: -50


**What I observed:** The int8 numbers stay between -128 and 127, and the values I get back are very close to the original but not exactly the same (small rounding error). 0.0 comes back as exactly 0.0.

## Task 2: Model size from parameter count

size = number of parameters x bytes per parameter (fp32 = 4 bytes, fp16 = 2 bytes, int8 = 1 byte). I used 1 GB = 1,000,000,000 bytes. This only counts the weights.

In [3]:
def model_size(params, dtype):
    bytes_per_param = {"fp32": 4, "fp16": 2, "int8": 1}
    total_bytes = params * bytes_per_param[dtype]
    return total_bytes / 1e6, total_bytes / 1e9

In [4]:
models = {"125M": 125e6, "1B": 1e9, "7B": 7e9}

for name, params in models.items():
    for dtype in ["fp32", "fp16", "int8"]:
        mb, gb = model_size(params, dtype)
        print(name, dtype, "->", mb, "MB =", gb, "GB")

125M fp32 -> 500.0 MB = 0.5 GB
125M fp16 -> 250.0 MB = 0.25 GB
125M int8 -> 125.0 MB = 0.125 GB
1B fp32 -> 4000.0 MB = 4.0 GB
1B fp16 -> 2000.0 MB = 2.0 GB
1B int8 -> 1000.0 MB = 1.0 GB
7B fp32 -> 28000.0 MB = 28.0 GB
7B fp16 -> 14000.0 MB = 14.0 GB
7B int8 -> 7000.0 MB = 7.0 GB


**What I observed:** Going from fp32 to int8 makes the model 4 times smaller (the 7B model goes from 28 GB to 7 GB). fp16 is half of fp32.

## Task 3: Multi-GPU inference design note

**Where to split:** I would use tensor parallelism inside one node (the 8 GPUs joined by NVLink) and pipeline parallelism across nodes (joined by InfiniBand). If I need to handle more users, I would run more copies of the whole setup.

**Why:** In tensor parallelism every layer is cut across the GPUs, and they have to share results many times for every token, so it needs a very fast link. In pipeline parallelism each node holds some layers and only passes a small output to the next node, so a slower link is fine.

**NVLink vs InfiniBand:** NVLink is the very fast link between GPUs inside a node. InfiniBand is the slower network between nodes. NVLink handles the heavy, frequent communication and InfiniBand handles the light, occasional communication.

**If I swapped them:** Tensor parallelism across nodes would make the GPUs wait on the slow network for every layer, so tokens would come out slowly. Pipeline parallelism inside the node would waste the fast NVLink on small transfers.

## Task 4: Compute-bound vs memory-bound

The GPU needs time to do the math (FLOPs / compute speed) and time to read the weights from memory (bytes / bandwidth). Whichever is bigger is the bottleneck. Weights are read once per step, but the math is done for every token in the batch, so I added a batch size.

In [5]:
def check(flops_per_token, gpu_flops, gpu_bandwidth, weight_bytes, batch_size):
    compute_time = flops_per_token * batch_size / gpu_flops
    memory_time = weight_bytes / gpu_bandwidth
    if compute_time > memory_time:
        return "compute-bound", compute_time, memory_time
    return "memory-bound", compute_time, memory_time

In [6]:
params = 7e9
flops_per_token = 2 * params
weight_bytes = params * 2

a100_flops, a100_bw = 312e12, 2.0e12
h100_flops, h100_bw = 989e12, 3.35e12

configs = [
    ("A100, batch 1", a100_flops, a100_bw, 1),
    ("A100, batch 256", a100_flops, a100_bw, 256),
    ("H100, batch 256", h100_flops, h100_bw, 256),
]

for name, f, bw, batch in configs:
    result, c_time, m_time = check(flops_per_token, f, bw, weight_bytes, batch)
    print(name, "->", result)
    print("   compute time:", round(c_time * 1000, 2), "ms, memory time:", round(m_time * 1000, 2), "ms")

A100, batch 1 -> memory-bound
   compute time: 0.04 ms, memory time: 7.0 ms
A100, batch 256 -> compute-bound
   compute time: 11.49 ms, memory time: 7.0 ms
H100, batch 256 -> memory-bound
   compute time: 3.62 ms, memory time: 4.18 ms


**What I observed:** At batch 1 the GPU is memory-bound, so making the model smaller (quantization) or using a bigger batch helps more than a faster GPU. At batch 256 on the A100 it is compute-bound, so quantization would not help much and more compute would. On the H100 the same batch is memory-bound again, because its compute grew faster than its memory speed.